# Data Collection and Panel Construction

This notebook builds the Syria host-country panel for the April 14 milestone of the DSA210 term project.

## Scope

- Origin country: Syria (`SYR`)
- Host countries: Turkey (`TUR`), Lebanon (`LBN`), Jordan (`JOR`)
- Time period: 2010–2024

## Data Sources

This notebook combines three official sources:

1. **UCDP Battle-Related Deaths Dataset**
   - used to measure annual conflict intensity in Syria
   - main variable: `bd_best`

2. **UNHCR Refugee Statistics API**
   - used to collect end-of-year refugee stocks by origin and host country
   - main variable used: `Refugees under UNHCR's mandate`

3. **World Bank World Development Indicators**
   - used to collect annual macroeconomic indicators for host countries

## Output

The final output of this notebook is a cleaned annual panel dataset saved as:

`data/processed/syria_panel_2010_2024.csv`

In [1]:
from pathlib import Path
import zipfile
import time
import requests
import pandas as pd
import numpy as np

YEAR_FROM = 2010
YEAR_TO = 2024

ORIGIN_ISO3 = "SYR"
HOSTS_ISO3 = ["TUR", "LBN", "JOR"]

REPO_ROOT = Path("..")
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUTS_DIR = REPO_ROOT / "outputs"
FIG_DIR = OUTPUTS_DIR / "figures"
TABLE_DIR = OUTPUTS_DIR / "tables"

for p in [RAW_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

UCDP_BRD_CONFLICT_ZIP_URL = "https://ucdp.uu.se/downloads/brd/ucdp-brd-conf-251-csv.zip"
UNHCR_POPULATION_URL = "https://api.unhcr.org/population/v1/population/"
WORLD_BANK_V2_BASE = "https://api.worldbank.org/v2"
import io

In [2]:
def _request_with_retries(method, url, *, params=None, headers=None, timeout=60, max_retries=5, backoff=2.0):
    attempt = 0
    while True:
        attempt += 1
        try:
            resp = requests.request(method, url, params=params, headers=headers, timeout=timeout)

            if resp.status_code in (429, 500, 502, 503, 504):
                if attempt >= max_retries:
                    resp.raise_for_status()
                time.sleep(backoff ** (attempt - 1))
                continue

            resp.raise_for_status()
            return resp

        except Exception:
            if attempt >= max_retries:
                raise
            time.sleep(backoff ** (attempt - 1))

## Conflict Intensity Data

This section downloads annual conflict intensity data for Syria from the UCDP Battle-Related Deaths Dataset.

The main conflict intensity variable used in this project is:

- `bd_best`: annual best estimate of battle-related deaths

This notebook aggregates annual Syria conflict intensity for 2010–2024 and creates the variable `conflict_bd_best_syria`.


In [3]:
ucdp_zip_path = RAW_DIR / "ucdp_brd_conf_251.zip"

if not ucdp_zip_path.exists():
    download_binary(UCDP_BRD_CONFLICT_ZIP_URL, ucdp_zip_path)

with zipfile.ZipFile(ucdp_zip_path, "r") as zf:
    csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    if not csv_names:
        raise ValueError("UCDP zip does not contain a CSV file.")
    ucdp_csv_name = sorted(csv_names)[0]
    with zf.open(ucdp_csv_name) as f:
        ucdp = pd.read_csv(f)

ucdp.columns = [c.strip() for c in ucdp.columns]

ucdp_syria = ucdp[
    (ucdp["year"] >= YEAR_FROM) &
    (ucdp["year"] <= YEAR_TO) &
    (ucdp["location_inc"].astype(str).str.strip().str.lower() == "syria")
].copy()

conflict_intensity = (
    ucdp_syria.groupby("year", as_index=False)["bd_best"]
    .sum()
    .rename(columns={"bd_best": "conflict_bd_best_syria"})
)

year_df = pd.DataFrame({"year": list(range(YEAR_FROM, YEAR_TO + 1))})
conflict_intensity = year_df.merge(conflict_intensity, on="year", how="left")
conflict_intensity["origin_iso"] = ORIGIN_ISO3
conflict_intensity["conflict_bd_best_syria"] = conflict_intensity["conflict_bd_best_syria"].fillna(0)

print(conflict_intensity)


    year  conflict_bd_best_syria origin_iso
0   2010                     0.0        SYR
1   2011                  1203.0        SYR
2   2012                 50490.0        SYR
3   2013                 72016.0        SYR
4   2014                 65345.0        SYR
5   2015                 50181.0        SYR
6   2016                 43650.0        SYR
7   2017                 24126.0        SYR
8   2018                 13072.0        SYR
9   2019                  7379.0        SYR
10  2020                  4589.0        SYR
11  2021                  1356.0        SYR
12  2022                   911.0        SYR
13  2023                  1500.0        SYR
14  2024                  2166.0        SYR


## Refugee Exposure Data

This section downloads refugee stock data from the UNHCR Refugee Statistics API.

The query is limited to:

- country of origin: Syria (`SYR`)
- host countries: Turkey, Lebanon, Jordan
- years: 2010–2024

The endpoint returns end-of-year population stock data. In this notebook, the main refugee exposure variable is defined as refugee stock rather than annual flow.

In [4]:
unhcr_params = {
    "download": "true",
    "cf_type": "ISO",
    "yearFrom": str(YEAR_FROM),
    "yearTo": str(YEAR_TO),
    "coo": ORIGIN_ISO3,
    "coa": ",".join(HOSTS_ISO3),
    "limit": "1000",
}

In [5]:
resp = _request_with_retries("GET", UNHCR_POPULATION_URL, params=unhcr_params)

base_name = f"unhcr_population_coo_{ORIGIN_ISO3}_hosts_{'_'.join(HOSTS_ISO3)}_{YEAR_FROM}_{YEAR_TO}"

if resp.content[:2] == b"PK":
    zip_path = RAW_DIR / f"{base_name}.zip"
    zip_path.write_bytes(resp.content)

    with zipfile.ZipFile(io.BytesIO(resp.content), "r") as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        xlsx_names = [n for n in zf.namelist() if n.lower().endswith(".xlsx") or n.lower().endswith(".xls")]

        if csv_names:
            with zf.open(csv_names[0]) as f:
                unhcr = pd.read_csv(f)
        elif xlsx_names:
            with zf.open(xlsx_names[0]) as f:
                unhcr = pd.read_excel(io.BytesIO(f.read()))
        else:
            raise ValueError("UNHCR zip did not contain a readable CSV or Excel file.")
else:
    csv_path = RAW_DIR / f"{base_name}.csv"
    csv_path.write_bytes(resp.content)
    unhcr = pd.read_csv(csv_path)

unhcr.columns = [c.strip() for c in unhcr.columns]

print(unhcr.columns.tolist())
print(unhcr.head(10))
print(unhcr.shape)


['Year', 'Country of origin', 'Country of asylum', 'Country of origin (ISO)', 'Country of asylum (ISO)', "Refugees under UNHCR's mandate", 'Asylum-seekers', 'Returned refugees', 'IDPs of concern to UNHCR', 'Returned IDPss', 'Stateless persons', 'Others of concern', 'Other people in need of international protection', 'Host Community']
   Year Country of origin Country of asylum Country of origin (ISO)  \
0  2010  Syrian Arab Rep.            Jordan                     SYR   
1  2011  Syrian Arab Rep.            Jordan                     SYR   
2  2012  Syrian Arab Rep.            Jordan                     SYR   
3  2013  Syrian Arab Rep.            Jordan                     SYR   
4  2014  Syrian Arab Rep.            Jordan                     SYR   
5  2015  Syrian Arab Rep.            Jordan                     SYR   
6  2016  Syrian Arab Rep.            Jordan                     SYR   
7  2017  Syrian Arab Rep.            Jordan                     SYR   
8  2018  Syrian Arab Rep.

In [6]:
unhcr_small = unhcr[[
    "Year",
    "Country of origin (ISO)",
    "Country of asylum (ISO)",
    "Refugees under UNHCR's mandate",
    "Asylum-seekers"
]].copy()

unhcr_small = unhcr_small.rename(columns={
    "Year": "year",
    "Country of origin (ISO)": "origin_iso",
    "Country of asylum (ISO)": "host_iso",
    "Refugees under UNHCR's mandate": "refugee_stock",
    "Asylum-seekers": "asylum_seekers"
})

unhcr_small["year"] = pd.to_numeric(unhcr_small["year"], errors="coerce").astype("Int64")
unhcr_small["origin_iso"] = unhcr_small["origin_iso"].astype(str).str.upper().str.strip()
unhcr_small["host_iso"] = unhcr_small["host_iso"].astype(str).str.upper().str.strip()

unhcr_small["refugee_stock"] = pd.to_numeric(unhcr_small["refugee_stock"], errors="coerce")
unhcr_small["asylum_seekers"] = pd.to_numeric(unhcr_small["asylum_seekers"], errors="coerce")

unhcr_small = unhcr_small[
    (unhcr_small["origin_iso"] == ORIGIN_ISO3) &
    (unhcr_small["host_iso"].isin(HOSTS_ISO3)) &
    (unhcr_small["year"] >= YEAR_FROM) &
    (unhcr_small["year"] <= YEAR_TO)
].copy()

unhcr_small = unhcr_small.sort_values(["host_iso", "year"]).reset_index(drop=True)

print(unhcr_small.shape)
print(unhcr_small.head(15))
print(unhcr_small.tail(15))
print(unhcr_small.isna().sum())

(45, 5)
    year origin_iso host_iso  refugee_stock  asylum_seekers
0   2010        SYR      JOR            198             287
1   2011        SYR      JOR            193            2618
2   2012        SYR      JOR         238798             491
3   2013        SYR      JOR         585304               0
4   2014        SYR      JOR         623112               0
5   2015        SYR      JOR         628223               0
6   2016        SYR      JOR         648836               0
7   2017        SYR      JOR         653031               0
8   2018        SYR      JOR         676283               0
9   2019        SYR      JOR         654692               0
10  2020        SYR      JOR         662790               0
11  2021        SYR      JOR         672952               0
12  2022        SYR      JOR         660892               0
13  2023        SYR      JOR         649091               0
14  2024        SYR      JOR         611473               0
    year origin_iso host_iso  re

## Macroeconomic Data

This section downloads host-country macroeconomic indicators from the World Bank World Development Indicators API.

Indicators used in the current panel:

- GDP growth
- inflation
- unemployment
- trade as a percentage of GDP
- current account balance as a percentage of GDP
- total population

These indicators are collected for Turkey, Lebanon, and Jordan for 2010–2024.

In [7]:
def worldbank_fetch_one(countries_iso3, indicator, year_from, year_to, per_page=20000):
    rows = []

    countries_seg = ";".join([c.lower() for c in countries_iso3])
    url = f"{WORLD_BANK_V2_BASE}/country/{countries_seg}/indicator/{indicator}"

    params = {
        "format": "json",
        "date": f"{year_from}:{year_to}",
        "per_page": per_page,
        "page": 1,
        "source": 2,
    }

    while True:
        resp = _request_with_retries(
            "GET",
            url,
            params=params,
            headers={"Accept": "application/json"},
            timeout=30,
            max_retries=3,
            backoff=1.5
        )

        payload = resp.json()

        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(f"Unexpected World Bank response for {indicator}")

        meta, data = payload[0], payload[1]

        if data is None:
            break

        for item in data:
            host_iso = item.get("countryiso3code")
            year_str = item.get("date")
            val = item.get("value", None)

            if host_iso is None or year_str is None:
                continue

            try:
                year = int(year_str)
            except Exception:
                continue

            rows.append({
                "host_iso": host_iso,
                "year": year,
                "indicator_code": indicator,
                "value": val
            })

        pages = int(meta.get("pages", 1))
        page = int(meta.get("page", 1))

        if page >= pages:
            break

        params["page"] = page + 1

    return pd.DataFrame(rows)


In [8]:
ALL_WDI = [
    "NY.GDP.MKTP.KD.ZG",
    "FP.CPI.TOTL.ZG",
    "SL.UEM.TOTL.ZS",
    "NE.TRD.GNFS.ZS",
    "BN.CAB.XOKA.GD.ZS",
    "SP.POP.TOTL",
]

wdi_frames = []

for ind in ALL_WDI:
    df_ind = worldbank_fetch_one(HOSTS_ISO3, ind, YEAR_FROM, YEAR_TO)
    wdi_frames.append(df_ind)

wdi_tidy = pd.concat(wdi_frames, ignore_index=True)

print(wdi_tidy.shape)
print(sorted(wdi_tidy["indicator_code"].dropna().unique().tolist()))
print(sorted(wdi_tidy["host_iso"].dropna().unique().tolist()))
print(wdi_tidy["year"].min(), wdi_tidy["year"].max())
print(wdi_tidy.head(20))


(270, 4)
['BN.CAB.XOKA.GD.ZS', 'FP.CPI.TOTL.ZG', 'NE.TRD.GNFS.ZS', 'NY.GDP.MKTP.KD.ZG', 'SL.UEM.TOTL.ZS', 'SP.POP.TOTL']
['JOR', 'LBN', 'TUR']
2010 2024
   host_iso  year     indicator_code      value
0       JOR  2024  NY.GDP.MKTP.KD.ZG   2.488894
1       JOR  2023  NY.GDP.MKTP.KD.ZG   2.884083
2       JOR  2022  NY.GDP.MKTP.KD.ZG   2.647435
3       JOR  2021  NY.GDP.MKTP.KD.ZG   3.655644
4       JOR  2020  NY.GDP.MKTP.KD.ZG  -1.102752
5       JOR  2019  NY.GDP.MKTP.KD.ZG   1.751241
6       JOR  2018  NY.GDP.MKTP.KD.ZG   1.919071
7       JOR  2017  NY.GDP.MKTP.KD.ZG   2.473598
8       JOR  2016  NY.GDP.MKTP.KD.ZG   1.994181
9       JOR  2015  NY.GDP.MKTP.KD.ZG   2.496529
10      JOR  2014  NY.GDP.MKTP.KD.ZG   3.384078
11      JOR  2013  NY.GDP.MKTP.KD.ZG   2.609947
12      JOR  2012  NY.GDP.MKTP.KD.ZG   2.429358
13      JOR  2011  NY.GDP.MKTP.KD.ZG   2.737180
14      JOR  2010  NY.GDP.MKTP.KD.ZG   2.314834
15      LBN  2024  NY.GDP.MKTP.KD.ZG        NaN
16      LBN  2023  NY.GDP.MKTP.

In [9]:
wdi_wide = (
    wdi_tidy
    .pivot_table(index=["host_iso", "year"], columns="indicator_code", values="value", aggfunc="first")
    .reset_index()
)

wdi_wide = wdi_wide.rename(columns={
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG": "inflation",
    "SL.UEM.TOTL.ZS": "unemployment",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "BN.CAB.XOKA.GD.ZS": "current_account_pct_gdp",
    "SP.POP.TOTL": "host_population",
})

for c in [
    "gdp_growth",
    "inflation",
    "unemployment",
    "trade_pct_gdp",
    "current_account_pct_gdp",
    "host_population",
]:
    if c in wdi_wide.columns:
        wdi_wide[c] = pd.to_numeric(wdi_wide[c], errors="coerce")

wdi_wide = wdi_wide.sort_values(["host_iso", "year"]).reset_index(drop=True)

print(wdi_wide.shape)
print(wdi_wide.head(15))
print(wdi_wide.isna().sum())

(45, 8)
indicator_code host_iso  year  current_account_pct_gdp  inflation  \
0                   JOR  2010                -6.935807   4.845519   
1                   JOR  2011               -10.012201   4.162442   
2                   JOR  2012               -14.892690   4.515230   
3                   JOR  2013               -10.169584   4.824623   
4                   JOR  2014                -7.077870   2.899479   
5                   JOR  2015                -8.992114  -0.876851   
6                   JOR  2016                -9.655514  -0.778430   
7                   JOR  2017               -10.557515   3.323894   
8                   JOR  2018                -6.831224   4.462311   
9                   JOR  2019                -1.736613   0.761514   
10                  JOR  2020                -5.732228   0.333294   
11                  JOR  2021                -8.030062   1.346094   
12                  JOR  2022                -8.112208   4.229156   
13                  JOR  2

## Panel Construction

This section merges the three components of the project dataset:

- annual Syria conflict intensity
- refugee stock by host country and year
- host-country macroeconomic indicators

## Merge Keys

- conflict data: `origin_iso + year`
- refugee data: `origin_iso + host_iso + year`
- macroeconomic data: `host_iso + year`

## Derived Variables

The notebook also constructs:

- `refugees_per_1000`
- `log_refugee_stock`

In [10]:
panel_index = pd.MultiIndex.from_product(
    [HOSTS_ISO3, range(YEAR_FROM, YEAR_TO + 1)],
    names=["host_iso", "year"]
)

panel = pd.DataFrame(index=panel_index).reset_index()
panel["origin_iso"] = ORIGIN_ISO3

panel = panel.merge(
    unhcr_small,
    on=["origin_iso", "host_iso", "year"],
    how="left"
)

panel = panel.merge(
    wdi_wide,
    on=["host_iso", "year"],
    how="left"
)

panel = panel.merge(
    conflict_intensity[["year", "origin_iso", "conflict_bd_best_syria"]],
    on=["year", "origin_iso"],
    how="left"
)

panel["refugees_per_1000"] = np.where(
    (panel["host_population"].notna()) &
    (panel["host_population"] > 0) &
    (panel["refugee_stock"].notna()),
    1000 * panel["refugee_stock"] / panel["host_population"],
    np.nan
)

panel["log_refugee_stock"] = np.log1p(panel["refugee_stock"])

HOST_NAME_MAP = {
    "TUR": "Turkey",
    "LBN": "Lebanon",
    "JOR": "Jordan",
}

panel["host_name"] = panel["host_iso"].map(HOST_NAME_MAP)
panel["origin_name"] = "Syria"

panel = panel[[
    "origin_name",
    "origin_iso",
    "host_name",
    "host_iso",
    "year",
    "conflict_bd_best_syria",
    "refugee_stock",
    "asylum_seekers",
    "host_population",
    "refugees_per_1000",
    "log_refugee_stock",
    "gdp_growth",
    "inflation",
    "unemployment",
    "trade_pct_gdp",
    "current_account_pct_gdp",
]].sort_values(["host_iso", "year"]).reset_index(drop=True)

print(panel.shape)
print(panel.head(15))
print(panel.isna().sum())

(45, 16)
   origin_name origin_iso host_name host_iso  year  conflict_bd_best_syria  \
0        Syria        SYR    Jordan      JOR  2010                     0.0   
1        Syria        SYR    Jordan      JOR  2011                  1203.0   
2        Syria        SYR    Jordan      JOR  2012                 50490.0   
3        Syria        SYR    Jordan      JOR  2013                 72016.0   
4        Syria        SYR    Jordan      JOR  2014                 65345.0   
5        Syria        SYR    Jordan      JOR  2015                 50181.0   
6        Syria        SYR    Jordan      JOR  2016                 43650.0   
7        Syria        SYR    Jordan      JOR  2017                 24126.0   
8        Syria        SYR    Jordan      JOR  2018                 13072.0   
9        Syria        SYR    Jordan      JOR  2019                  7379.0   
10       Syria        SYR    Jordan      JOR  2020                  4589.0   
11       Syria        SYR    Jordan      JOR  2021     

## Final Check

The cleaned Syria host-country panel has now been created and saved.

This dataset will be used in the second notebook for:

- exploratory data analysis
- missingness assessment
- hypothesis testing

Final output file:

`data/processed/syria_panel_2010_2024.csv`

In [11]:
processed_path = PROCESSED_DIR / "syria_panel_2010_2024.csv"
panel.to_csv(processed_path, index=False)

print(processed_path)

../data/processed/syria_panel_2010_2024.csv


In [12]:
missing_summary = panel.isna().sum().reset_index()
missing_summary.columns = ["variable", "missing_count"]
missing_summary["missing_rate"] = missing_summary["missing_count"] / len(panel)

print(missing_summary)

                   variable  missing_count  missing_rate
0               origin_name              0      0.000000
1                origin_iso              0      0.000000
2                 host_name              0      0.000000
3                  host_iso              0      0.000000
4                      year              0      0.000000
5    conflict_bd_best_syria              0      0.000000
6             refugee_stock              0      0.000000
7            asylum_seekers              0      0.000000
8           host_population              0      0.000000
9         refugees_per_1000              0      0.000000
10        log_refugee_stock              0      0.000000
11               gdp_growth              1      0.022222
12                inflation              0      0.000000
13             unemployment              1      0.022222
14            trade_pct_gdp              1      0.022222
15  current_account_pct_gdp              1      0.022222


In [13]:
print(panel.shape)
print(panel.groupby("host_iso")["year"].count())
print(panel.isna().sum())
print(panel[["host_iso", "year", "refugee_stock", "host_population", "refugees_per_1000"]].head(20))

(45, 16)
host_iso
JOR    15
LBN    15
TUR    15
Name: year, dtype: int64
origin_name                0
origin_iso                 0
host_name                  0
host_iso                   0
year                       0
conflict_bd_best_syria     0
refugee_stock              0
asylum_seekers             0
host_population            0
refugees_per_1000          0
log_refugee_stock          0
gdp_growth                 1
inflation                  0
unemployment               1
trade_pct_gdp              1
current_account_pct_gdp    1
dtype: int64
   host_iso  year  refugee_stock  host_population  refugees_per_1000
0       JOR  2010            198        7297043.0           0.027134
1       JOR  2011            193        7480424.0           0.025801
2       JOR  2012         238798        7587127.0          31.474101
3       JOR  2013         585304        7991809.0          73.237987
4       JOR  2014         623112        8791710.0          70.874949
5       JOR  2015         628223    